In [10]:
!gdown --folder https://drive.google.com/drive/folders/1BC6JqmBTdIisdcA8iEoir4n7Zvw2o8xX

Retrieving folder contents
Processing file 1z0r60MX5cjvwS0XMxFi9SJJedH5zJgG8 all_results.csv
Processing file 1kEevVzYXhZ1gwjWyyL3__sYgfVuWeKE1 results_maec16_vol_only.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1z0r60MX5cjvwS0XMxFi9SJJedH5zJgG8
To: /content/ecc-volatility-prediction-results-for-ablation/all_results.csv
100% 52.9k/52.9k [00:00<00:00, 66.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1kEevVzYXhZ1gwjWyyL3__sYgfVuWeKE1
To: /content/ecc-volatility-prediction-results-for-ablation/results_maec16_vol_only.csv
100% 5.58k/5.58k [00:00<00:00, 12.0MB/s]
Download completed


In [2]:
import pandas as pd
df_r = pd.read_csv('ecc-volatility-prediction-results-for-ablation/all_results.csv')
df_rv = pd.read_csv('ecc-volatility-prediction-results-for-ablation/results_maec16_vol_only.csv')

df1 = df_r.copy()

df1["experiment"] = df1["modality"].astype(str) + "_" + df1["fusion_type"].astype(str)

df2 = df_rv.copy()
df2["experiment"] = "volonly"

metrics = ["mse_3", "mse_7", "mse_15", "mse_30", "avg_mse"]

df1 = df1[["seed", "experiment"] + metrics]
df2 = df2[["seed", "experiment"] + metrics]

df_combined = pd.concat([df1, df2], ignore_index=True)

df = df_combined.pivot(
    index="seed",
    columns="experiment",
    values=metrics
)

df.columns = [
    f"{metric}_{experiment}"
    for metric, experiment in df.columns
]

df = df.reset_index()
df.columns.name = None

print(df.shape)
df.head()

(55, 41)


,seed,mse_3_audio_only_residual_gated,mse_3_chunk_text_audio_residual_gated,mse_3_text_audio_prompt1_residual_gated,mse_3_text_audio_prompt2_concat,mse_3_text_audio_prompt2_qa_only,mse_3_text_audio_prompt2_residual_gated,mse_3_text_only_residual_gated,mse_3_volonly,mse_7_audio_only_residual_gated,...,mse_30_text_only_residual_gated,mse_30_volonly,avg_mse_audio_only_residual_gated,avg_mse_chunk_text_audio_residual_gated,avg_mse_text_audio_prompt1_residual_gated,avg_mse_text_audio_prompt2_concat,avg_mse_text_audio_prompt2_qa_only,avg_mse_text_audio_prompt2_residual_gated,avg_mse_text_only_residual_gated,avg_mse_volonly
0,1,0.337162,0.325204,0.328047,0.334635,0.349171,0.328050,0.327987,0.367623,0.192616,...,0.146836,0.141552,0.224659,0.220556,0.220003,0.248684,0.246219,0.220055,0.219817,0.228630
1,3,0.328786,0.324242,0.328987,0.384017,0.404336,0.328921,0.328811,0.338035,0.177196,...,0.127474,0.146804,0.209379,0.209830,0.209822,0.247050,0.264193,0.209614,0.209403,0.220226
2,5,0.330861,0.325370,0.330777,0.373480,0.366932,0.330868,0.330875,0.348633,0.190160,...,0.131791,0.147418,0.212354,0.209878,0.212042,0.272569,0.236919,0.212310,0.212172,0.225160
3,6,0.329933,0.329290,0.338019,0.330103,0.412696,0.332547,0.349410,0.334525,0.178735,...,0.152853,0.139918,0.212532,0.215268,0.220592,0.237604,0.281867,0.215457,0.229226,0.218214
4,9,0.327978,0.320764,0.327709,0.385681,0.418214,0.327543,0.327401,0.335552,0.182921,...,0.135832,0.180791,0.214503,0.210688,0.214044,0.268608,0.282608,0.213741,0.213426,0.236945


In [7]:
import numpy as np
from scipy import stats

def summarize(df, winner, others):
    """Print mean MSE (sorted) and winner-vs-other paired-t results."""
    cols = [winner] + others
    print(f"n = {len(df)} seeds")
    print()
    print("Mean, sorted (lower = better):")
    print(df[cols].mean().sort_values())
    print()
    print(f"{'model':30s} {'mean diff':>12s} {'p-value':>10s}")
    for m in others:
        d = (df[m] - df[winner]).values
        t, p_two = stats.ttest_1samp(d, 0)
        p_one = p_two / 2 if t > 0 else 1 - p_two / 2
        print(f"{m:30s} {d.mean():12.5f} {p_one:10.4f}")

In [12]:
for h in [3, 7, 15, 30]:
    winner = f"mse_{h}_text_audio_prompt2_residual_gated"

    others = [
        f"mse_{h}_text_audio_prompt1_residual_gated",
        f"mse_{h}_audio_only_residual_gated",
        f"mse_{h}_text_only_residual_gated",
        f"mse_{h}_chunk_text_audio_residual_gated",
        f"mse_{h}_text_audio_prompt2_concat",
        f"mse_{h}_text_audio_prompt2_qa_only",
        f"mse_{h}_volonly",
    ]

    print(f"\n{'=' * 20} Horizon {h} {'=' * 20}")
    summarize(df, winner, others)


winner = "avg_mse_text_audio_prompt2_residual_gated"

others = [
    "avg_mse_text_audio_prompt1_residual_gated",
    "avg_mse_audio_only_residual_gated",
    "avg_mse_text_only_residual_gated",
    "avg_mse_chunk_text_audio_residual_gated",
    "avg_mse_text_audio_prompt2_concat",
    "avg_mse_text_audio_prompt2_qa_only",
    "avg_mse_volonly",
]

print("\n" + "=" * 20 + " Average MSE " + "=" * 20)
summarize(df, winner, others)


==================== Horizon 3 ====================
n = 55 seeds

Mean, sorted (lower = better):
mse_3_chunk_text_audio_residual_gated      0.331129
mse_3_text_audio_prompt2_residual_gated    0.336699
mse_3_text_audio_prompt1_residual_gated    0.338362
mse_3_audio_only_residual_gated            0.338395
mse_3_text_only_residual_gated             0.339502
mse_3_volonly                              0.348018
mse_3_text_audio_prompt2_concat            0.352176
mse_3_text_audio_prompt2_qa_only           0.391396
dtype: float64

model                             mean diff    p-value
mse_3_text_audio_prompt1_residual_gated      0.00166     0.1214
mse_3_audio_only_residual_gated      0.00170     0.0501
mse_3_text_only_residual_gated      0.00280     0.1999
mse_3_chunk_text_audio_residual_gated     -0.00557     0.9999
mse_3_text_audio_prompt2_concat      0.01548     0.0013
mse_3_text_audio_prompt2_qa_only      0.05470     0.0000
mse_3_volonly                       0.01132     0.0003

=========

### Pairwise comparison of all models

All-pairs paired one-sided t-test between models.

In [13]:
import numpy as np
import pandas as pd
from scipy import stats

model_suffixes = [
    "text_audio_prompt1_residual_gated",
    "text_audio_prompt2_residual_gated",
    "audio_only_residual_gated",
    "text_only_residual_gated",
    "chunk_text_audio_residual_gated",
    "text_audio_prompt2_concat",
    "text_audio_prompt2_qa_only",
    "volonly",
]

metrics = [f"mse_{h}" for h in [3, 7, 15, 30]] + ["avg_mse"]

for metric in metrics:
    all_models = [f"{metric}_{suffix}" for suffix in model_suffixes]

    short_names = {
        m: m.replace(f"{metric}_", "")
        for m in all_models
    }

    mean_diff = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    p_value = pd.DataFrame(index=all_models, columns=all_models, dtype=float)

    for row in all_models:
        for col in all_models:
            if row == col:
                mean_diff.loc[row, col] = 0.0
                p_value.loc[row, col] = np.nan
                continue

            # Positive means the row model has lower MSE (better)
            d = (df[col] - df[row]).values
            t, p_two = stats.ttest_1samp(d, 0, nan_policy="raise")
            p_one = p_two / 2 if t > 0 else 1 - p_two / 2

            mean_diff.loc[row, col] = d.mean()
            p_value.loc[row, col] = p_one

    mean_diff = mean_diff.rename(index=short_names, columns=short_names)
    p_value = p_value.rename(index=short_names, columns=short_names)

    print(f"\n{'=' * 25} {metric.upper()} {'=' * 25}")
    print(f"n = {len(df)} seeds\n")

    print("Mean difference (row model improvement over column model):")
    print(mean_diff.round(5))
    print()

    print("One-sided p-value (row model is better than column model):")
    print(p_value.round(4))

n = 55 seeds

Mean difference (row model improvement over column model):
                                   text_audio_prompt1_residual_gated  \
text_audio_prompt1_residual_gated                            0.00000   
text_audio_prompt2_residual_gated                            0.00155   
audio_only_residual_gated                                   -0.00055   
text_only_residual_gated                                    -0.00237   
chunk_text_audio_residual_gated                              0.00330   
text_audio_prompt2_concat                                   -0.02679   
text_audio_prompt2_qa_only                                  -0.04554   
volonly                                                     -0.01522   

                                   text_audio_prompt2_residual_gated  \
text_audio_prompt1_residual_gated                           -0.00155   
text_audio_prompt2_residual_gated                            0.00000   
audio_only_residual_gated                                   -0